# GeoParquet: Vector Data Goes Cloud-Native
### From bbox covering to native Parquet geometry types — where things stand in 2026

*Marc Weber*

## Why this talk

A few things converged in the last year:

- Apache **Parquet** shipped native `GEOMETRY` / `GEOGRAPHY` logical types (Parquet 2.11, March 2025) — geo is no longer bolted on
- **GeoParquet 2.0.0-rc.1** re-aligns the spec around those native types
- Real, huge, public datasets (Overture Maps) now ship *only* as GeoParquet
- A new generation of tools (`geoparquet-io`, GeoLibre, DuckDB-WASM) treat GeoParquet as a first-class citizen

We'll write some, inspect some, query some over HTTP, and load some into a map.

## Agenda

1. Cloud-optimized, for vectors — the COG analogy
2. GeoParquet 101, hands-on (write it, inspect it)
3. What changed in 2026: Parquet's native geo types → GeoParquet 2.0
4. The ecosystem check: what actually supports what, today
5. Live query: Overture Maps GeoParquet straight from S3
6. Application: **GeoLibre**, a cloud-native GIS that speaks GeoParquet
7. Cheat sheet + resources

# 1 · Cloud-optimized, for vectors

## The COG playbook, applied to vectors

Cloud-Optimized GeoTIFF made rasters *queryable in place*: internal tiling + overviews + HTTP range requests meant a client could read a small window of a huge file without downloading it whole.

GeoParquet applies the same idea to vector/tabular geo data:

- **Columnar layout** — read only the columns (attributes) you need
- **Row groups** — each holds min/max statistics, so a client can skip whole chunks that can't match a filter (predicate pushdown)
- **bbox column + `covering` metadata** — a per-row bounding box column lets engines prune on *space*, using the same statistics mechanism
- **HTTP range requests** — cloud object stores serve just the byte ranges a query engine asks for

## GeoParquet is a thin, well-behaved layer on Parquet

It is **not** a new file format. It's:

- an agreement on how to store geometries in a Parquet column (WKB, or native GeoArrow-encoded types), and
- a small JSON blob in the Parquet file's key/value metadata (key `"geo"`) describing CRS, geometry types, bbox, and (as of 1.1) the `covering` column

Because it *is* Parquet, every Parquet-speaking tool already sees the data — DuckDB, Spark, Arrow, pandas, Snowflake, BigQuery. Geo-awareness is opt-in on top.

# 2 · GeoParquet 101, hands-on

## Write a tiny GeoParquet file

A handful of Pacific Northwest cities, written with GeoPandas 1.x — note `write_covering_bbox`, the 1.1 feature that adds a per-row bbox struct column for cheap spatial pruning.

In [2]:
import geopandas as gpd
from shapely.geometry import Point

data = {
    "city": ["Corvallis", "Portland", "Bend", "Eugene", "Astoria"],
    "state": ["OR"] * 5,
    "population": [59_000, 652_000, 100_000, 176_000, 10_000],
    "geometry": [
        Point(-123.2620, 44.5646),
        Point(-122.6765, 45.5231),
        Point(-121.3153, 44.0582),
        Point(-123.0868, 44.0521),
        Point(-123.8313, 46.1879),
    ],
}
gdf = gpd.GeoDataFrame(data, crs="EPSG:4326")

gdf.to_parquet(
    "pnw_cities.parquet",
    write_covering_bbox=True,   # GeoParquet 1.1: per-row bbox struct column
    schema_version="1.1.0",     # be explicit; default is still "1.0.0"
)
gdf

,city,state,population,geometry
0,Corvallis,OR,59000,POINT (-123.262 44.5646)
1,Portland,OR,652000,POINT (-122.6765 45.5231)
2,Bend,OR,100000,POINT (-121.3153 44.0582)
3,Eugene,OR,176000,POINT (-123.0868 44.0521)
4,Astoria,OR,10000,POINT (-123.8313 46.1879)


## What actually landed in the file

Parquet's *file* metadata carries a `"geo"` key with the GeoParquet metadata JSON — what a reader checks before touching a single row.

In [3]:
import json
import pyarrow.parquet as pq

meta = pq.read_metadata("pnw_cities.parquet")
kv = dict(meta.metadata or {})
geo = json.loads(kv[b"geo"])

print("schema version :", geo["version"])
print("geometry types :", geo["columns"]["geometry"]["geometry_types"])
print("column bbox    :", geo["columns"]["geometry"]["bbox"])
print("covering       :", geo["columns"]["geometry"].get("covering"))

schema version : 1.1.0
geometry types : ['Point']
column bbox    : [-123.8313, 44.0521, -121.3153, 46.1879]
covering       : {'bbox': {'xmin': ['bbox', 'xmin'], 'ymin': ['bbox', 'ymin'], 'xmax': ['bbox', 'xmax'], 'ymax': ['bbox', 'ymax']}}


In [4]:
pq.read_schema("pnw_cities.parquet")

city: large_string
state: large_string
population: int64
geometry: binary
  -- field metadata --
  ARROW:extension:name: 'geoarrow.wkb'
  ARROW:extension:metadata: '{"crs": {"$schema": "https://proj.org/schema' + 1498
bbox: struct<xmin: double, ymin: double, xmax: double, ymax: double>
  child 0, xmin: double
  child 1, ymin: double
  child 2, xmax: double
  child 3, ymax: double
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 715
geo: '{"primary_column": "geometry", "columns": {"geometry": {"encoding":' + 1491

Notice the extra `bbox` struct column (`xmin`/`ymin`/`xmax`/`ymax`) sitting right next to `geometry` — that's what a query engine's row-group and page-index statistics latch onto for spatial pruning, with zero geometry parsing required.

# 3 · What changed in 2026

## Recap: GeoParquet 1.0 → 1.1

GeoParquet 1.1 (the version most tools write today) added, on top of 1.0:

- **`covering` metadata** pointing at a bbox struct column, for cheap spatial predicate pushdown without a spatial index
- Stricter **geometry-type declarations** — every type actually present must be listed
- Optional **native GeoArrow encodings** as an alternative to WKB

## The bigger shift: Parquet gets geometry *natively*

In March 2025, **Apache Parquet 2.11** added `GEOMETRY` and `GEOGRAPHY` as first-class **logical types** — part of the format spec itself:

- Values stored as **WKB** in a `BYTE_ARRAY` column
- `GEOMETRY` = planar (projected CRS); `GEOGRAPHY` = spherical/ellipsoidal (lon/lat, great-circle semantics)
- Parquet row groups now carry **native min/max bbox statistics** for the geometry column
- CRS metadata (EPSG code or PROJJSON) travels on the logical type
- Early support: Parquet Java, Arrow C++/Rust, DuckDB, Hyparquet (JS), and **GDAL/OGR 3.12**

> Chris Holmes (GeoParquet co-creator): *"2026 will be the year where we really start to see major datasets use Parquet `GEOMETRY` and `GEOGRAPHY` types."*

## GeoParquet 2.0.0-rc.1: converging, not competing

GeoParquet 2.0 re-anchors itself on Parquet's native types:

- Geometry columns **must** use the Parquet `GEOMETRY`/`GEOGRAPHY` logical type on a `BYTE_ARRAY` field (WKB) — GeoArrow-native encodings are dropped
- **CRS now lives on the logical type's `crs` property** — the source of truth; the GeoParquet `"geo"` JSON just restates it for older readers
- Geometry columns must sit at the **schema root** (no nesting)
- A writer without PROJJSON support can emit native types with just an authority code (e.g. `EPSG:4326`) and skip GeoParquet metadata entirely

One practitioner's framing (rednegra.net) is worth keeping in mind: *a Parquet file with a `GEOMETRY` column is not automatically a GeoParquet file* — GeoParquet still defines a **primary** geometry column, mandatory per-column geometry-type lists, polygon winding metadata, CRS epochs, and (for now) the `covering` convention.

## Where the *tools* actually are, today

Verified against what's installed in this environment right now:

In [5]:
import geopandas, pyarrow
from geopandas.io.arrow import SUPPORTED_VERSIONS

print("geopandas   :", geopandas.__version__)
print("pyarrow     :", pyarrow.__version__)
print("schema_version options geopandas can write:", SUPPORTED_VERSIONS)

geopandas   : 1.1.4
pyarrow     : 25.0.0
schema_version options geopandas can write: ['0.1.0', '0.4.0', '1.0.0-beta.1', '1.0.0', '1.1.0']


GeoPandas can *write* GeoParquet 1.1 metadata (bbox covering) today, but doesn't yet emit Parquet's native `GEOMETRY`/`GEOGRAPHY` logical type or GeoParquet 2.0. GDAL/OGR (≥3.12) is ahead of the Python geo stack here — pin `schema_version` explicitly rather than assume "latest."

## `geoparquet-io`: an opinionated toolbelt

A newer (2026, beta) CLI + Python library purpose-built for this transition:

- One-command optimization: bbox column + **Hilbert-curve sort** + ZSTD + smart row-group sizing — reported 10–100× query speedups over naive writes
- Converts Shapefile / GeoJSON / GeoPackage → optimized GeoParquet
- Reads/writes S3, GCS, Azure directly, with credential auto-discovery
- Validates spec compliance and can repair non-compliant files in place
- Partitioning strategies for big datasets: H3, S2, quadkey, admin boundaries

In [5]:
# illustrative — geoparquet-io is in beta
# pip install geoparquet-io
# gpio convert cities.shp cities.parquet --optimize
# gpio validate cities.parquet

# 4 · Live: Overture Maps over HTTP

## The dataset: Overture Maps, GeoParquet-only

[Overture Maps](https://overturemaps.org) (Linux Foundation: Amazon, Meta, Microsoft, TomTom, and others) publishes global places, buildings, transportation, and administrative boundaries **as GeoParquet on public S3**, hive-partitioned by theme/type, no auth required:

```
s3://overturemaps-us-west-2/release/{release}/theme={theme}/type={type}/*
```

This is the dataset that makes "cloud-optimized" concrete: there is no download-the-file-first step.

## Query it in place with DuckDB

*(needs internet access — DuckDB fetches the `spatial`/`httpfs` extensions on first use, then reads directly from S3)*

In [6]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("SET s3_region='us-west-2';")

RELEASE = "2026-08-19.0"  # check https://docs.overturemaps.org for the current release
AOI = dict(xmin=-123.35, xmax=-123.20, ymin=44.53, ymax=44.60)  # Corvallis, OR

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## Places, filtered by bbox — pushed down, not post-filtered

In [7]:
places = con.execute(f"""
    SELECT id, names.primary AS name, categories.primary AS category, geometry
    FROM read_parquet(
        's3://overturemaps-us-west-2/release/{RELEASE}/theme=places/type=place/*',
        filename = true, hive_partitioning = 1
    )
    WHERE bbox.xmin BETWEEN {AOI['xmin']} AND {AOI['xmax']}
      AND bbox.ymin BETWEEN {AOI['ymin']} AND {AOI['ymax']}
    LIMIT 25
""").df()
places

,id,name,category,geometry
0,abe1fa18-3653-4811-9cb4-6307066d5e32,Holden Hair Salon,hair_salon,"[1, 1, 0, 0, 0, 130, 114, 120, 148, 12, 205, 9..."
1,e672383a-9a5e-48ff-8454-3a97ba59b095,Placid Construction,contractor,"[1, 1, 0, 0, 0, 143, 210, 58, 228, 69, 205, 94..."
2,6dbc5aa5-4236-4630-8642-8f21eec9a7ed,Tanuki Interactive,web_designer,"[1, 1, 0, 0, 0, 109, 55, 193, 55, 77, 205, 94,..."
3,56f565b9-2f2d-4581-ac38-994ddeae58a9,Pacific Stonescape,nursery_and_gardening,"[1, 1, 0, 0, 0, 171, 122, 249, 157, 38, 205, 9..."
4,9bdcc3d9-b3d2-4db6-9b6b-429c9b5ee408,Great Internet Results LLC,advertising_agency,"[1, 1, 0, 0, 0, 238, 255, 255, 255, 24, 205, 9..."
5,23c2769c-3071-45f0-aa81-1822702cb8cb,Porter Remodeling Inc,NaN,"[1, 1, 0, 0, 0, 157, 53, 46, 210, 250, 204, 94..."
6,ceecaba2-821f-4b5b-b20a-9e16a6df166a,Willamette Tours and Cruises,tours,"[1, 1, 0, 0, 0, 57, 110, 8, 192, 5, 205, 94, 1..."
7,8bef3125-13cd-4194-a2d1-0898e37865b4,RJH Enterprises,automotive_repair,"[1, 1, 0, 0, 0, 44, 157, 15, 207, 18, 205, 94,..."
8,dd5ed38a-9a7c-45ca-8a4b-5dd565815084,Peoria Road Farm Market,farmers_market,"[1, 1, 0, 0, 0, 99, 235, 124, 236, 172, 205, 9..."
9,01c75f3e-f3b9-4f99-b9c8-3ded7448d1d9,Blue Heron Farm,nursery_and_gardening,"[1, 1, 0, 0, 0, 242, 196, 222, 81, 132, 205, 9..."


The `bbox.xmin` / `bbox.ymin` predicate is exactly the GeoParquet 1.1 `covering` column at work: DuckDB skips entire row groups using their statistics before it ever decodes a WKB geometry.

## Buildings, same AOI — export straight to a normal GIS format

In [8]:
con.execute(f"""
    COPY (
        SELECT id, names.primary AS name, height, geometry
        FROM read_parquet(
            's3://overturemaps-us-west-2/release/{RELEASE}/theme=buildings/type=building/*',
            filename = true, hive_partitioning = 1
        )
        WHERE bbox.xmin BETWEEN {AOI['xmin']} AND {AOI['xmax']}
          AND bbox.ymin BETWEEN {AOI['ymin']} AND {AOI['ymax']}
    ) TO 'corvallis_buildings.geojson' WITH (FORMAT GDAL, DRIVER 'GeoJSON')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

No shapefile-era download-the-whole-country step. One SQL query, scoped by a bounding box, against a live cloud dataset measured in hundreds of gigabytes.

# 5 · Application: GeoLibre

## A cloud-native GIS that speaks GeoParquet natively

[**GeoLibre**](https://geolibre.app) ([opengeos/GeoLibre](https://github.com/opengeos/GeoLibre)) is a lightweight, open-source (MIT) GIS that runs — from one codebase — in the browser, on desktop, on mobile, and **inside Jupyter and Quarto/R**:

- **1,000+ geoprocessing tools**, running client-side via WebAssembly — no server, no install for the browser build
- Built on **MapLibre GL JS**, **DuckDB-WASM (spatial)**, and **deck.gl** for rendering/SQL, **Tauri v2** for the native desktop/mobile shell
- First-class conversion to/from the cloud-native trio: **GeoParquet, PMTiles, and COG**
- Ships as a **Python package** (Jupyter, `leafmap`-style API) and an **R package** (`geolibre-r`, for RStudio, Quarto, Shiny)

## Loading GeoParquet into GeoLibre, from Python

In [9]:
from geolibre import Map

m = Map(center=(-123.0, 44.5), zoom=7, basemap="dark", height="500px")

# local file written earlier in this notebook
m.add_geoparquet(
    "pnw_cities.parquet",
    name="PNW Cities",
    paint={"circle-color": "#f2a900", "circle-radius": 6},
)

m

## Buildings we read as geojson file with SQL

`add_geojson()` takes a URL just as happily as a local path — a GeoLibre map and the DuckDB query above are really two views onto the same underlying cloud-optimized file:

In [16]:
m2 = Map(center=(-123.26, 44.56), zoom=12, basemap="positron")
m2.add_geojson(
    "corvallis_buildings.geojson",
    name="Overture Buildings (Corvallis)",
    paint={"fill-color": "#4c78a8", "fill-opacity": 0.6},
)
m2

In the R/Quarto world, the equivalent is `geolibre-r`'s widget — same underlying MapLibre + DuckDB-WASM engine, so the same GeoParquet URL drops straight into an RStudio/Quarto session too.

# 6 · Cheat sheet

## Tool → what it's good for, right now

| Tool | Reads native Parquet `GEOMETRY`? | Writes GeoParquet | Notes |
|---|---|---|---|
| GDAL / OGR ≥ 3.12 | ✅ | ✅ (1.x + native types) | furthest along |
| DuckDB (`spatial` ext.) | ✅ | ✅ | great for ad-hoc cloud queries |
| GeoPandas 1.1.x | ❌ (WKB only) | ✅ up to 1.1.0 | `schema_version=` to pin |
| Apache Sedona | ✅ (Spark-scale) | ✅ | best for very large batch jobs |
| `geoparquet-io` | via GDAL/DuckDB | ✅ (optimized) | beta; validate + optimize |
| GeoLibre | via DuckDB-WASM | — (viewer/editor) | browser/desktop/Jupyter/Quarto |

## Summary

- GeoParquet changed the paradigm of "download the whole vector data" into "query the bytes you need" — the same shift as COG for rasters
- 2026 is the year that shift moves **into Parquet itself**: native `GEOMETRY`/`GEOGRAPHY` types, and GeoParquet 2.0 rebuilt on top of them
- The Python geo stack (GeoPandas) is a step behind GDAL/DuckDB on native types today
- Overture Maps is the dataset to practice on: real scale, no authentication, no download required
- GeoLibre is a nice end-to-end demo of "cloud-native geo" as a *product*, not just a file format

## Resources

- GeoParquet spec — https://geoparquet.org · [1.1.0](https://geoparquet.org/releases/v1.1.0/) · [2.0.0-rc.1](https://geoparquet.org/releases/v2.0.0-rc.1/)
- Apache Parquet native geo types — https://parquet.apache.org/blog/2026/02/13/native-geospatial-types-in-apache-parquet/
- Chris Holmes, *"GeoParquet & Parquet geospatial types: A time of transition"* — https://medium.com/radiant-earth-insights/geoparquet-parquet-geospatial-types-a-time-of-transition-a42e391cdab2
- *"Parquet with GEOMETRY type is not GeoParquet"* — https://rednegra.net/blog/20250925-parquet-with-geometry-type-is-not-geoparquet/
- `geoparquet-io` — https://cloudnativegeo.org/blog/2026/03/introducing-geoparquet-io/ · https://geoparquet.io/CHANGELOG/
- Overture Maps + DuckDB — https://docs.overturemaps.org/getting-data/duckdb/
- GeoLibre — https://geolibre.app · https://github.com/opengeos/GeoLibre · R package: https://r.geolibre.app